In [6]:
!pip install asyncio
!pip install playwright

   ---------------------------------------- 0.0/36.8 MB ? eta -:--:--
   -------- ------------------------------- 8.1/36.8 MB 50.4 MB/s eta 0:00:01
   ------------------- -------------------- 18.4/36.8 MB 48.2 MB/s eta 0:00:01
   ----------------------------- ---------- 27.3/36.8 MB 46.7 MB/s eta 0:00:01
   ---------------------------------------  35.9/36.8 MB 45.6 MB/s eta 0:00:01
   ---------------------------------------- 36.8/36.8 MB 43.3 MB/s eta 0:00:00


In [ ]:
import asyncio
import csv
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright

PROFILE_URL = "https://truthsocial.com/@realDonaldTrump"

async def main():
    results = []

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()

        await page.goto(PROFILE_URL, wait_until="networkidle", timeout=120000)

        last_height = 0
        stagnant = 0

        # Scroll until no more content loads a few times in a row
        for _ in range(200):
            html = await page.content()
            soup = BeautifulSoup(html, "html.parser")

            # You will likely need to inspect the live DOM and adjust selectors
            # depending on Truth Social's frontend markup.
            for article in soup.select("article"):
                text = article.get_text("\n", strip=True)
                link = article.find("a", href=True)
                href = link["href"] if link else ""
                results.append({"text": text, "href": href})

            await page.evaluate("window.scrollTo(0, document.body.scrollHeight)")
            await page.wait_for_timeout(2000)

            new_height = await page.evaluate("document.body.scrollHeight")
            if new_height == last_height:
                stagnant += 1
            else:
                stagnant = 0
            last_height = new_height

            if stagnant >= 5:
                break

        await browser.close()

    # dedupe
    deduped = []
    seen = set()
    for row in results:
        key = (row["text"], row["href"])
        if key not in seen:
            seen.add(key)
            deduped.append(row)

    with open("donald_trump_truths_scraped.csv", "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["text", "href"])
        writer.writeheader()
        writer.writerows(deduped)

    print(f"Saved {len(deduped)} rows")

asyncio.run(main())

c:\Users\huste\anaconda3\lib\ast.py:50: RuntimeWarning: coroutine 'main' was never awaited
  return compile(source, filename, mode, flags,


NotImplementedError: 